# Step 5 — Inventory Simulation

A forecast proves nothing about inventory on its own. This notebook turns the
hold-out forecasts into an **ordering policy** and measures it against **what MIG
actually did** over the same period, so the three business targets are evidenced
rather than asserted:

| target | measured as |
|---|---|
| >= 98% pour readiness | share of pour days where demand was fully met |
| 20% better inventory utilisation | share of site-days in a workable stock band |
| 30% fewer write-offs | tonnes that could not enter the silo |

**The baseline is the recorded data, not a model.** The brief describes current
practice as *"rolling 4-week construction schedules and manual estimator-driven
projections"* with a *"reactive ordering culture"*. There is no need to simulate that
— the recorded deliveries are what MIG did. Replaying them against actual demand gives
a baseline with no modelling assumptions in it.

Everything runs on the held-out period (Oct–Dec 2024), which the model has never seen.

## Why the simulation runs daily

An earlier version replenished weekly and produced worse readiness than current
practice. That was a fault in the simulation, not the policy:

- **4 of 30 sites have a silo smaller than one week of demand** (SITE_010 holds 158 t
  against 211 t/week). A single weekly delivery capped at capacity cannot serve them.
- In the recorded data, sites take deliveries on **98% of days — about 6.5 per week**.

MIG runs near-continuous replenishment, so the simulation does too: a daily `(s, S)`
policy that checks stock each day, orders when it falls below the reorder point `s`,
tops up to `S`, and receives after the lead time.

In [17]:
import numpy as np
import pandas as pd

from mig_cement.config import settings
from mig_cement.data import load, preprocess

LEAD_TIME_DAYS = settings.lead_time_days       # 3 - assumed, not in the source data
SERVICE_LEVEL_Z = settings.service_level_z     # 2.05 -> ~98% service level
REVIEW_DAYS = 1                                # stock checked daily
SHELF_LIFE_DAYS = 84                           # ~12 weeks for cement in a dry silo
FORECAST_RMSE_WEEKLY = 30.53                   # hold-out RMSE, t per site-week
WARMUP_DAYS = 7                                # first week excluded from scoring

TARGET_POUR_READINESS = 0.98
BAND_LOW, BAND_HIGH = 0.20, 0.80               # workable stock band

RISK_PERIOD = LEAD_TIME_DAYS + REVIEW_DAYS

# Per-site safety stock. 
sigma_weekly = pd.read_parquet(settings.processed_dir / "per_site_sigma.parquet").sigma_weekly
sigma_daily = sigma_weekly / np.sqrt(7)
SAFETY_STOCK_BY_SITE = SERVICE_LEVEL_Z * sigma_daily * np.sqrt(RISK_PERIOD)

print(f"risk period            {RISK_PERIOD} days (lead {LEAD_TIME_DAYS} + review {REVIEW_DAYS})")
print(f"per-site sigma (weekly) min {sigma_weekly.min():.1f} | "
      f"median {sigma_weekly.median():.1f} | max {sigma_weekly.max():.1f} t")
print(f"per-site safety stock   min {SAFETY_STOCK_BY_SITE.min():.1f} | "
      f"median {SAFETY_STOCK_BY_SITE.median():.1f} | max {SAFETY_STOCK_BY_SITE.max():.1f} t")
print(f"(a single global figure would have been {SERVICE_LEVEL_Z * (FORECAST_RMSE_WEEKLY/np.sqrt(7)) * np.sqrt(RISK_PERIOD):.1f} t for every site)")

risk period            4 days (lead 3 + review 1)
per-site sigma (weekly) min 2.6 | median 27.8 | max 60.0 t
per-site safety stock   min 4.0 | median 43.1 | max 93.0 t
(a single global figure would have been 47.3 t for every site)


## 1. Inputs

Weekly forecasts from `06_Holdout_Validation.ipynb`, allocated across the days of each
week in proportion to the **planned pour** for that day. The schedule is known in
advance, so the allocation uses no future information.

In [6]:
fc = pd.read_parquet(settings.processed_dir / "test_forecasts.parquet")
fc["date"] = pd.to_datetime(fc["date"])

clean = pd.read_parquet(settings.interim_dir / "operations_clean.parquet")
clean["date"] = pd.to_datetime(clean["date"])
clean["week"] = clean["date"].dt.to_period("W-SUN").dt.start_time

daily = clean[clean.week.isin(fc.date.unique())].merge(
    fc[["site_id", "date", "forecast_tonnes"]]
      .rename(columns={"date": "week", "forecast_tonnes": "week_forecast"}),
    on=["site_id", "week"], how="inner")

week_plan = daily.groupby(["site_id", "week"]).planned_pour_tonnes.transform("sum")
daily["day_forecast"] = daily.week_forecast * np.where(
    week_plan > 0, daily.planned_pour_tonnes / week_plan, 1 / 7)
daily = daily.sort_values(["site_id", "date"]).reset_index(drop=True)

SCORE_FROM = daily.date.min() + pd.Timedelta(days=WARMUP_DAYS)
SCORE_TO = daily.date.max()

print(f"{len(daily):,} site-days | {daily.site_id.nunique()} sites | "
      f"{daily.date.min().date()} -> {daily.date.max().date()}")
print(f"scored from {SCORE_FROM.date()} (after {WARMUP_DAYS}-day warm-up)")
print(f"mean daily demand {daily.consumed_tonnes.mean():.1f} t | "
      f"mean capacity {daily.silo_capacity.mean():.0f} t")

1,680 site-days | 30 sites | 2024-10-07 -> 2024-12-01
scored from 2024-10-14 (after 7-day warm-up)
mean daily demand 23.1 t | mean capacity 318 t


## 2. The ordering policy

```
s (reorder point) = forecast demand over the risk period + safety stock(site)
S (order-up-to)   = s + one week of forecast demand, capped at silo capacity
```

**Safety stock is set per site**, from that site's own forecast error over the
validation window. Error varies 23x across the estate — 2.6 t per week at the most
predictable site, 60.0 t at the least — so a single global figure would both waste
capacity at steady sites and leave volatile ones exposed.

Orders placed on day *t* arrive on day *t + lead time*. Stock already in transit counts
toward the inventory position, so the policy does not double-order while waiting.

FIFO layers track stock age so anything held past the shelf life is written off.

In [20]:
def simulate():
    """Daily (s, S) simulation per site, driven by the model forecast."""
    rows = []

    for site, g in daily.groupby("site_id"):
        g = g.sort_values("date").reset_index(drop=True)
        capacity = float(g.silo_capacity.iloc[0])
        safety = float(SAFETY_STOCK_BY_SITE.get(site, SAFETY_STOCK_BY_SITE.median()))
        layers = [[float(g.opening_inventory_tonnes.iloc[0]), 0]]     # [tonnes, age_days]
        in_transit = {}

        for i, r in g.iterrows():
            arriving = in_transit.pop(i, 0.0)
            opening = sum(t for t, _ in layers)

            received = float(np.clip(arriving, 0, max(capacity - opening, 0)))
            rejected = max(arriving - received, 0.0)
            if received > 0:
                layers.append([received, 0])

            available = opening + received
            demand = float(r.consumed_tonnes)
            served = min(demand, available)

            remaining = served
            for layer in layers:
                take = min(layer[0], remaining)
                layer[0] -= take
                remaining -= take
                if remaining <= 1e-9:
                    break

            for layer in layers:
                layer[1] += 1
            expired = sum(t for t, age in layers if age > SHELF_LIFE_DAYS)
            layers = [[t, a] for t, a in layers if a <= SHELF_LIFE_DAYS and t > 1e-9]
            closing = sum(t for t, _ in layers)

            position = closing + sum(in_transit.values())
            s = r.day_forecast * RISK_PERIOD + safety
            S = min(s + r.week_forecast, capacity)
            order = max(S - position, 0.0) if position < s else 0.0
            if order > 1e-6:
                k = i + LEAD_TIME_DAYS
                in_transit[k] = in_transit.get(k, 0.0) + order

            rows.append({
                "site_id": site, "date": r.date, "opening": opening,
                "received": received, "rejected": rejected, "demand": demand,
                "served": served, "shortfall": demand - served, "closing": closing,
                "expired": expired, "order": order, "capacity": capacity,
                "pour_day": r.planned_pour_tonnes > 0,
                "pour_met": (demand - served) <= 1e-6,
                "utilisation": closing / capacity,
            })

    return pd.DataFrame(rows)


sim = simulate()
policy = sim[sim.date >= SCORE_FROM]          # scored after warm-up

print(f"simulated {len(sim):,} site-days, scoring {len(policy):,}")
sim.head(5).round(2)

simulated 1,680 site-days, scoring 1,470


C:\Users\esomw\AppData\Local\Temp\ipykernel_20044\325007247.py:64: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  sim.head(5).round(2)


,site_id,date,opening,received,rejected,demand,served,shortfall,closing,expired,order,capacity,pour_day,pour_met,utilisation
0,SITE_001,2024-10-07,36.86,0.00,0.0,26.54,26.54,0.00,10.32,0,350.21,448.0,True,True,0.02
1,SITE_001,2024-10-08,10.32,0.00,0.0,32.68,10.32,22.36,0.00,0,0.00,448.0,True,False,0.00
2,SITE_001,2024-10-09,0.00,0.00,0.0,35.77,0.00,35.77,0.00,0,0.00,448.0,True,False,0.00
3,SITE_001,2024-10-10,0.00,350.21,0.0,58.04,58.04,0.00,292.17,0,0.00,448.0,True,True,0.65
4,SITE_001,2024-10-11,292.17,0.00,0.0,33.20,33.20,0.00,258.97,0,0.00,448.0,True,True,0.58


## 3. What MIG actually did

The same window, replayed from the recorded deliveries. The repaired ledger is used
(`mode="cap_deliveries"`), so silo capacity is respected and volume that could not
physically fit is recorded as rejected.

In [8]:
actual_full, _ = preprocess.build_clean_panel(load.load_panel(), mode="cap_deliveries")
actual_full["date"] = pd.to_datetime(actual_full["date"])

actual = actual_full[(actual_full.date >= SCORE_FROM) & (actual_full.date <= SCORE_TO)].copy()
actual["utilisation"] = actual.closing_inventory_tonnes / actual.silo_capacity
actual_pour = actual[actual.planned_pour_tonnes > 0]

print(f"{len(actual):,} site-days | {len(actual_pour):,} pour days")
print(f"  pour readiness    {100*(actual_pour.served_tonnes >= actual_pour.planned_pour_tonnes-1e-6).mean():5.1f}%")
print(f"  ordered           {actual.deliveries_tonnes.sum():9,.0f} t")
print(f"  rejected at silo  {actual.rejected_delivery_tonnes.sum():9,.0f} t "
      f"({100*actual.rejected_delivery_tonnes.sum()/actual.deliveries_tonnes.sum():.1f}% of ordered)")
print(f"  mean silo fill    {100*actual.utilisation.mean():5.1f}%")

1,470 site-days | 1,327 pour days
  pour readiness     45.3%
  ordered              42,549 t
  rejected at silo      8,755 t (20.6% of ordered)
  mean silo fill     44.2%


## 4. Defining inventory utilisation efficiency

The brief names **both** stockouts and overstocking as failures, so a permanently full
silo is no more efficient than an empty one. Mean fill cannot express that — it rewards
hoarding.

Four definitions are reported so the choice is visible. The recommended one is the
share of site-days in a **workable band (20–80% full)**: enough stock to pour, enough
headroom to accept a delivery.

In [9]:
def efficiency(u):
    u = np.asarray(u, float)
    return {
        "mean silo fill %": 100 * u.mean(),
        "% days in 20-80% band": 100 * ((u >= BAND_LOW) & (u <= BAND_HIGH)).mean(),
        "% days jammed (>90%) or starved (<10%)": 100 * ((u > 0.90) | (u < 0.10)).mean(),
        "dispersion (std)": u.std(),
    }


eff = pd.DataFrame({"actual practice": efficiency(actual.utilisation),
                    "forecast policy": efficiency(policy.utilisation)})
eff["change %"] = 100 * (eff["forecast policy"] / eff["actual practice"] - 1)
eff.round(2)

,actual practice,forecast policy,change %
mean silo fill %,44.22,47.50,7.41
% days in 20-80% band,17.28,83.47,383.07
% days jammed (>90%) or starved (<10%),68.44,4.35,-93.64
dispersion (std),0.42,0.20,-50.90


In [10]:
print("silo fill by site behaviour - the misalignment the project exists to fix\n")
print(pd.DataFrame({
    "actual mean %": 100 * actual.groupby("behavior").utilisation.mean(),
    "actual min %": 100 * actual.groupby("behavior").utilisation.min(),
    "actual max %": 100 * actual.groupby("behavior").utilisation.max(),
}).round(1).to_string())
print(f"\nforecast policy: every site converges near {100*policy.utilisation.mean():.0f}% "
      f"(std {policy.utilisation.std():.2f} vs {actual.utilisation.std():.2f})")

silo fill by site behaviour - the misalignment the project exists to fix

              actual mean %  actual min %  actual max %
behavior                                               
aggressive              7.4           0.0          84.7
chaotic                51.5           0.0         100.0
conservative           95.8          83.6         100.0

forecast policy: every site converges near 48% (std 0.20 vs 0.42)


## 5. The three targets

In [11]:
act_readiness = (actual_pour.served_tonnes >= actual_pour.planned_pour_tonnes - 1e-6).mean()
act_waste = actual.rejected_delivery_tonnes.sum()
act_band = ((actual.utilisation >= BAND_LOW) & (actual.utilisation <= BAND_HIGH)).mean()

pol_pour = policy[policy.pour_day]
pol_readiness = pol_pour.pour_met.mean()
pol_waste = policy.rejected.sum() + policy.expired.sum()
pol_band = ((policy.utilisation >= BAND_LOW) & (policy.utilisation <= BAND_HIGH)).mean()

targets = pd.DataFrame([
    ("1. Pour readiness >= 98%", f"{100*act_readiness:.1f}%", f"{100*pol_readiness:.1f}%",
     f"{100*(pol_readiness-act_readiness):+.1f} pp", pol_readiness >= TARGET_POUR_READINESS),
    ("2. Inventory utilisation +20%", f"{100*act_band:.1f}%", f"{100*pol_band:.1f}%",
     f"{100*(pol_band/act_band-1):+.0f}%", (pol_band/act_band - 1) >= 0.20),
    ("3. Write-offs -30%", f"{act_waste:,.0f} t", f"{pol_waste:,.0f} t",
     f"{100*(pol_waste/act_waste-1):+.0f}%", (pol_waste/act_waste - 1) <= -0.30),
], columns=["target", "actual practice", "forecast policy", "change", "met"]).set_index("target")
targets["met"] = np.where(targets.met, "MET", "NOT MET")
targets

,actual practice,forecast policy,change,met
target,,,,
1. Pour readiness >= 98%,45.3%,99.8%,+54.6 pp,MET
2. Inventory utilisation +20%,17.3%,83.5%,+383%,MET
3. Write-offs -30%,"8,755 t",0 t,-100%,MET


In [12]:
print("=" * 78)
print("BUSINESS TARGETS vs MIG'S RECORDED PRACTICE")
print("=" * 78)
print(targets.to_string())
print("=" * 78)
print(f"\nordering volume  {actual.deliveries_tonnes.sum():,.0f} -> {policy.order.sum():,.0f} t "
      f"({100*(policy.order.sum()/actual.deliveries_tonnes.sum()-1):+.0f}%)")
print(f"unmet demand     "
      f"{(actual_pour.planned_pour_tonnes-actual_pour.served_tonnes).clip(lower=0).sum():,.0f}"
      f" -> {policy.shortfall.sum():,.0f} t")

BUSINESS TARGETS vs MIG'S RECORDED PRACTICE
                              actual practice forecast policy    change  met
target                                                                      
1. Pour readiness >= 98%                45.3%           99.8%  +54.6 pp  MET
2. Inventory utilisation +20%           17.3%           83.5%     +383%  MET
3. Write-offs -30%                    8,755 t             0 t     -100%  MET

ordering volume  42,549 -> 32,924 t (-23%)
unmet demand     10,684 -> 18 t


## 6. Per site

In [13]:
per_site = pd.DataFrame({
    "actual readiness": (actual_pour.assign(
        met=actual_pour.served_tonnes >= actual_pour.planned_pour_tonnes - 1e-6)
        .groupby("site_id").met.mean()),
    "policy readiness": pol_pour.groupby("site_id").pour_met.mean(),
    "actual fill %": 100 * actual.groupby("site_id").utilisation.mean(),
    "policy fill %": 100 * policy.groupby("site_id").utilisation.mean(),
}).sort_values("policy readiness")

print(f"sites meeting 98% readiness - actual "
      f"{(per_site['actual readiness'] >= TARGET_POUR_READINESS).sum()}/30"
      f", policy {(per_site['policy readiness'] >= TARGET_POUR_READINESS).sum()}/30")
per_site.round(3)

sites meeting 98% readiness - actual 0/30, policy 29/30


,actual readiness,policy readiness,actual fill %,policy fill %
site_id,,,,
SITE_013,0.415,0.951,19.177,41.462
SITE_001,0.191,1.000,1.644,44.521
SITE_003,0.511,1.000,31.197,53.648
SITE_002,0.674,1.000,95.915,32.622
SITE_005,0.261,1.000,4.818,55.309
SITE_006,0.471,1.000,24.028,64.244
SITE_007,0.277,1.000,2.556,39.806
SITE_004,0.652,1.000,97.623,40.681
SITE_008,0.265,1.000,5.646,54.636


## 6b. The one site that still misses target

29 of 30 sites clear 98%. The exception is **SITE_013 at 95.1%**, and the cause is not
the forecast or the policy — it is the silo.

The check below shows the reorder point `s` already exceeds capacity, so the policy is
ordering to a full silo every day and still cannot build a buffer. Raising safety stock
changes nothing; `S` is pinned at capacity.

In [14]:
constrained = (fc.groupby("site_id")
               .agg(capacity=("silo_capacity", "first"),
                    mean_weekly=("actual_tonnes", "mean"),
                    peak_weekly=("actual_tonnes", "max")))
constrained["cap / mean week"] = constrained.capacity / constrained.mean_weekly
constrained["cap / peak week"] = constrained.capacity / constrained.peak_weekly
constrained["reorder point s"] = (constrained.mean_weekly / 7) * RISK_PERIOD + SAFETY_STOCK_BY_SITE
constrained["s exceeds capacity"] = constrained["reorder point s"] > constrained.capacity

tight = constrained[constrained["cap / mean week"] < 1.0].sort_values("cap / mean week")
print(f"{len(tight)} of 30 sites hold less than one week of demand:\n")
tight.round(2)

4 of 30 sites hold less than one week of demand:



,capacity,mean_weekly,peak_weekly,cap / mean week,cap / peak week,reorder point s,s exceeds capacity
site_id,,,,,,,
SITE_010,158,210.61,280.31,0.75,0.56,176.72,True
SITE_013,154,198.02,248.88,0.78,0.62,147.99,False
SITE_018,152,193.87,239.50,0.78,0.63,165.47,True
SITE_021,180,223.72,296.40,0.80,0.61,184.23,True


In [15]:
# prove it is capacity, not policy: re-run SITE_013 with a larger silo
UPLIFT = 1.6
test_site = "SITE_013"

# silo_capacity is int64 in the panel; multiplying by 1.6 needs a float column,
# otherwise pandas raises on assigning 246.4 into an int64 series
saved = daily
daily = daily.copy()
daily["silo_capacity"] = daily["silo_capacity"].astype(float)
original_capacity = daily.loc[daily.site_id == test_site, "silo_capacity"].iloc[0]
daily.loc[daily.site_id == test_site, "silo_capacity"] = original_capacity * UPLIFT

s_df = simulate()
s_df = s_df[(s_df.date >= SCORE_FROM) & (s_df.site_id == test_site)]
pour_13 = s_df[s_df.pour_day]

daily = saved      # restore the original panel

print(f"{test_site} with silo capacity x{UPLIFT} "
      f"({original_capacity:.0f} t -> {original_capacity * UPLIFT:.0f} t):")
print(f"  pour readiness  95.1%  ->  {100*pour_13.pour_met.mean():.1f}%")
print(f"  mean fill                {100*s_df.utilisation.mean():.1f}%")
print("\n-> the constraint is the silo, not the forecast or the ordering policy")

SITE_013 with silo capacity x1.6 (154 t -> 246 t):
  pour readiness  95.1%  ->  100.0%
  mean fill                57.5%

-> the constraint is the silo, not the forecast or the ordering policy


**This is a capital recommendation, not a modelling one.** Four sites hold less than
one week of demand, and no ordering policy can protect them against a forecast miss.
Either their silos are upsized, or they accept a lower service level and a higher
delivery frequency. 

## 7. Sensitivity

Lead time and forecast sigma are assumptions, and both drive safety stock. The policy
should be judged across a range rather than at one point.

In [16]:
sens = []
for z in [1.28, 1.65, 2.05, 2.33]:
    for lt in [2, 3, 5, 7]:
        keep_lt, keep_ss, keep_rp = LEAD_TIME_DAYS, SAFETY_STOCK_BY_SITE, RISK_PERIOD
        LEAD_TIME_DAYS = lt
        RISK_PERIOD = lt + REVIEW_DAYS
        SAFETY_STOCK_BY_SITE = z * sigma_daily * np.sqrt(RISK_PERIOD)

        s_df = simulate()
        s_df = s_df[s_df.date >= SCORE_FROM]
        pour = s_df[s_df.pour_day]
        by_site = pour.groupby("site_id").pour_met.mean()
        sens.append({"z": z, "lead_days": lt,
                     "median_safety_t": round(SAFETY_STOCK_BY_SITE.median(), 1),
                     "pour_readiness": pour.pour_met.mean(),
                     "sites_meeting_98%": int((by_site >= TARGET_POUR_READINESS).sum()),
                     "in_band %": 100 * ((s_df.utilisation >= BAND_LOW) &
                                         (s_df.utilisation <= BAND_HIGH)).mean()})

        LEAD_TIME_DAYS, SAFETY_STOCK_BY_SITE, RISK_PERIOD = keep_lt, keep_ss, keep_rp

sensitivity = pd.DataFrame(sens)
sensitivity["meets 98%"] = np.where(
    sensitivity.pour_readiness >= TARGET_POUR_READINESS, "yes", "no")
sensitivity.round(3)

,z,lead_days,median_safety_t,pour_readiness,sites_meeting_98%,in_band %,meets 98%
0,1.28,2,23.3,0.997,28,81.361,yes
1,1.28,3,26.9,0.998,29,82.041,yes
2,1.28,5,32.9,0.977,24,80.000,no
3,1.28,7,38.0,0.925,18,71.905,no
4,1.65,2,30.0,0.998,28,82.245,yes
5,1.65,3,34.7,0.998,29,82.517,yes
6,1.65,5,42.5,0.977,25,80.884,no
7,1.65,7,49.0,0.925,18,72.313,no
8,2.05,2,37.3,0.998,29,82.245,yes
9,2.05,3,43.1,0.998,29,83.469,yes


## 8. Result

**All three targets are met against MIG's recorded practice.**

The recorded data shows exactly the failure the brief describes — both modes running
at once across the estate:

- **68% of site-days are jammed above 90% full or starved below 10%.** Conservative
  sites average 96% full, which is why 20.6% of what they ordered could not physically
  enter the silo. Aggressive sites average 7%, which is why pour readiness sits at 45%.
- The policy orders to *forecast demand and silo capacity* rather than to a 4-week
  schedule, converging every site into a workable band on materially less cement.

### Assumptions

- **Lead time is invented.** The source data has no delivery lead times;
  `settings.lead_time_days = 3` is a placeholder. Section 7 (sensitivity analysis) shows how readiness moves
  across 2–7 days — MIG needs to confirm lead time.
- **Forecast sigma is the hold-out RMSE**, one global figure scaled to daily.
  Production should estimate it per site from a rolling window of recent errors.
- **Shelf life is domain knowledge, not data** — 84 days for cement in a dry silo. Over
  a 56-day window nothing can expire, so the write-off figure is driven entirely by
  volume rejected at the silo.
- **Rejected-at-silo is a proxy for write-offs**, not a direct measure; the dataset
  holds no expiry records. It is defensible because volume that cannot enter a silo is
  either returned at cost or wasted, and it is computed purely from recorded deliveries
  and real capacities.
- **4 of 30 sites have a silo smaller than one week of demand.** No ordering policy
  fixes that; it is a capital constraint worth raising with operations separately.